# T16 CUDA throughput smoke-03

Owner-run throughput evidence only. This notebook does not run a quality gate or serving qualification. Choose a Colab GPU runtime, then use **Runtime → Run all**. Recommended: **A100** for this smoke (the learner is small and launch-overhead-bound, so the GPU model barely matters; the vCPU count does, and the A100 runtime has as many vCPUs as the H100 at roughly two thirds of the unit rate). Pick H100 only if the A100 result shows GPU utilization above about 70% on the self-play rows. L4 works but has fewer vCPUs; TPU and CPU-only runtimes are not supported. Expect up to 75 minutes of runtime: an unscored cache warm-up, a 60 s cProfile capture, and four 600 s scored self-play rows. It writes resumable artifacts and `colab_throughput_results.json` to Drive even when preflight fails.

In [ ]:
# Parameters. Change RUN_NAME only to start a separate, non-resumed smoke.
PINNED_COMMIT = 'fd3f0fc69214a6f42db591061ee9f5142f1d6e3a'
RUN_NAME = 'smoke-03'
DRIVE_ROOT = '/content/drive/MyDrive/t16_colab'
INPUT_ARCHIVE = f'{DRIVE_ROOT}/t16_colab_inputs.tar.gz'
EXPECTED_ARCHIVE_SHA256 = '229be2949c99f198b27decdf403b83ec4085d8df2dcfe629b7eff9208593be8e'
RESULTS_DIR = f'{DRIVE_ROOT}/{RUN_NAME}'
REPOSITORY = 'https://github.com/josenunezwork/snake-dqn.git'
CUDA_WHEEL_INDEX = 'https://download.pytorch.org/whl/cu126'
TOTAL_BUDGET_SECONDS = 4500  # setup, parity, warm-up, 60 s profile, four 600 s timings; runner cap is 4500
TIMING_SECONDS = 600
SCRATCH_DIR = '/content/t16-colab-scratch'  # local disk for trainer outputs; copied to Drive per run


In [ ]:
import hashlib
import json
import os
import signal
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive, files

drive.mount('/content/drive')
started_after_drive_auth = time.monotonic()
drive_root = Path(DRIVE_ROOT)
results_dir = Path(RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
result_path = results_dir / 'colab_throughput_results.json'
invocation_path = results_dir / 'invocation_result.json'
report_path = result_path
deadline = started_after_drive_auth + TOTAL_BUDGET_SECONDS

def cleanup_owned_group(process, grace_seconds):
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    stop_at = min(deadline, time.monotonic() + grace_seconds)
    while time.monotonic() < stop_at:
        process.poll()
        try:
            os.killpg(process.pid, 0)
        except ProcessLookupError:
            return
        time.sleep(0.1)
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

def run_owned(command, *, check=True, capture_output=False, cleanup_grace=5, **kwargs):
    timeout = deadline - time.monotonic() - cleanup_grace
    if timeout <= 0:
        raise TimeoutError(f'The {TOTAL_BUDGET_SECONDS}-second budget expired before this subprocess could start.')
    print('+', ' '.join(map(str, command)), flush=True)
    if capture_output:
        kwargs['stdout'] = subprocess.PIPE
    process = subprocess.Popen(command, text=True, start_new_session=True, **kwargs)
    try:
        stdout, _ = process.communicate(timeout=timeout)
    except BaseException as error:
        cleanup_owned_group(process, cleanup_grace)
        try:
            process.wait(timeout=max(0.1, min(1, deadline - time.monotonic())))
        except subprocess.TimeoutExpired:
            raise RuntimeError(f'Owned subprocess did not exit after cleanup: {command[0]}') from error
        if isinstance(error, subprocess.TimeoutExpired):
            raise TimeoutError(f'Subprocess exceeded the total {TOTAL_BUDGET_SECONDS}-second budget: {command[0]}') from error
        raise
    cleanup_owned_group(process, cleanup_grace)
    completed = subprocess.CompletedProcess(command, process.returncode, stdout=stdout)
    if check and completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command, output=stdout)
    return completed

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

interpreter_info = {}
try:
    archive = Path(INPUT_ARCHIVE)
    if not archive.is_file():
        raise FileNotFoundError(f'Missing Drive input archive: {archive}')
    if sha256(archive) != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError('Input archive SHA-256 differs from the published bundle; re-upload it unchanged.')

    checkout = Path('/content/t16-snake-dqn')
    venv = Path('/content/t16-colab-venv')
    shutil.rmtree(checkout, ignore_errors=True)
    shutil.rmtree(venv, ignore_errors=True)
    run_owned(['git', 'init', str(checkout)])
    run_owned(['git', '-C', str(checkout), 'remote', 'add', 'origin', REPOSITORY])
    run_owned(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT])
    run_owned(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'])
    actual_commit = run_owned(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], capture_output=True).stdout.strip()
    if actual_commit != PINNED_COMMIT:
        raise RuntimeError(f'Pinned checkout mismatch: expected {PINNED_COMMIT}, got {actual_commit}')

    # The 3.13 notebook kernel only orchestrates; uv supplies the workload interpreter.
    sys.path.insert(0, str(checkout))
    from research.t16_colab.environment import create_environment
    python, interpreter_info = create_environment(venv, run_owned)
    # requirements-gpu carries the repo pins; keep NumPy/Numba ABI pins explicit.
    run_owned([python, '-m', 'pip', 'install', '-r', str(checkout / 'requirements-gpu.txt'),
                 'numpy==1.26.4', 'numba==0.68.0', 'llvmlite==0.50.0', 'pytest>=9,<10'])
    # A fresh 3.12 venv cannot reuse the kernel's 3.13 torch wheel.
    run_owned([python, '-m', 'pip', 'install', '--index-url', CUDA_WHEEL_INDEX,
                 'torch==2.9.1'])
    run_owned([python, '-c',
        "import torch; assert torch.__version__.split('+')[0].startswith('2.9.') and torch.cuda.is_available(), torch.__version__"])

    extracted_parent = Path('/content/t16-colab-extracted')
    shutil.rmtree(extracted_parent, ignore_errors=True)
    verifier = (
        'from pathlib import Path; from research.t16_colab.bundle import verify_and_extract; '
        'print(verify_and_extract(Path(' + repr(str(archive)) + '), Path(' + repr(str(extracted_parent)) + ')))'
    )
    run_owned([python, '-c', verifier], cwd=checkout)
    input_root = extracted_parent / 't16-colab-inputs'
    remaining_budget = int(deadline - time.monotonic() - 12)
    if remaining_budget <= 0:
        raise TimeoutError(f'The {TOTAL_BUDGET_SECONDS}-second budget expired during setup.')
    run = [python, 'research/t16_colab/run_smoke.py', '--inputs', str(input_root),
           '--output', str(results_dir), '--commit', PINNED_COMMIT, '--suite', 'smoke03', '--seconds', str(TIMING_SECONDS),
           '--budget-seconds', str(remaining_budget), '--scratch-dir', SCRATCH_DIR]
    invocation_path.unlink(missing_ok=True)
    completed = run_owned(run, cwd=checkout, check=False, cleanup_grace=12)
    if completed.returncode and not invocation_path.exists():
        raise RuntimeError(f'runner exited {completed.returncode} before writing its invocation result')
    report_path = Path('/content/t16-colab-current/colab_throughput_results.json')
    report_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(invocation_path, report_path)
except Exception as error:
    failure = {
        'schema': 't16-colab-throughput/v1', 'commit': PINNED_COMMIT, 'configs': [],
        **interpreter_info, 'kernel_python': sys.version,
        'status': 'failed', 'error': f'{type(error).__name__}: {error}',
        'authority': 'throughput smoke only; not gate, quality, or serving evidence',
        'invocation_wall_seconds': time.monotonic() - started_after_drive_auth,
    }
    result_path = Path('/content/t16-colab-failure/colab_throughput_results.json')
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_path.write_text(json.dumps(failure, indent=2) + '\n')
    report_path = result_path
    print(failure['error'], file=sys.stderr)
finally:
    print(f'Results: {report_path}')
    if report_path.exists():
        display(json.loads(report_path.read_text()))
        files.download(str(report_path))
